In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import cv2

# -----------------------------
# Dataset (MNIST -> 32x32 input)
# -----------------------------
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

trainset = torchvision.datasets.MNIST(root='./data', train=True,
                                      download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True)

testset = torchvision.datasets.MNIST(root='./data', train=False,
                                     download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False)

classes = [str(i) for i in range(10)]

# -----------------------------
# Modified LeNet for CAM (16x16 feature maps, 10 channels)
# -----------------------------
class LeNet_CAM(nn.Module):
    def __init__(self, num_classes=10):
        super(LeNet_CAM, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=5, padding=2)   # -> 16x32x32
        self.pool1 = nn.AvgPool2d(2, 2)                           # -> 16x16x16
        self.conv2 = nn.Conv2d(16, 10, kernel_size=3, padding=1)  # -> 10x16x16 (final conv layer)
        self.gap = nn.AdaptiveAvgPool2d(1)                        # -> 10x1x1
        self.fc = nn.Linear(10, num_classes)                      # classifier

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)
        x = F.relu(self.conv2(x))
        fmap = x                          # last conv feature maps (10x16x16)
        x = self.gap(fmap)
        x = x.view(-1, 10)
        out = self.fc(x)
        return out, fmap

# -----------------------------
# Training setup
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LeNet_CAM().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# -----------------------------
# Training loop
# -----------------------------
epochs = 2
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        outputs, _ = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # Evaluate after each epoch
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs, _ = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(trainloader):.4f}, Test Acc: {acc:.2f}%")

torch.save(model.state_dict(), "lenet_cam_16x16.pth")
print("Model saved as lenet_cam_16x16.pth")

# -----------------------------
# CAM computation
# -----------------------------
def compute_cam(feature_maps, fc_weights, class_idx, input_size=(32, 32)):
    fmap = feature_maps[0].detach().cpu().numpy()          # [C, h, w] = [10,16,16]
    weights = fc_weights[class_idx].detach().cpu().numpy() # [C] = [10]

    # Weighted sum
    cam = np.dot(weights, fmap.reshape(fmap.shape[0], -1)) # [h*w]
    cam = cam.reshape(fmap.shape[1], fmap.shape[2])

    # ReLU
    cam = np.maximum(cam, 0)

    # Normalize
    if cam.max() > cam.min():
        cam = (cam - cam.min()) / (cam.max() - cam.min())

    # Resize to input size
    cam = cv2.resize(cam, input_size, interpolation=cv2.INTER_LINEAR)
    return cam

# -----------------------------
# Run CAM on a test image
# -----------------------------

model.eval()
images, labels = next(iter(testloader))
images, labels = images.to(device), labels.to(device)

outputs, fmap = model(images[10].unsqueeze(0))
pred_class = outputs.argmax(dim=1).item()

fc_weights = model.fc.weight.data  # [10, 10]

# Compute CAM
cam = compute_cam(fmap, fc_weights, pred_class, input_size=(32, 32))

# -----------------------------
# Plot CAM overlay
# -----------------------------
img = images[10].cpu().squeeze().numpy()

plt.figure(figsize=(10,4))
plt.subplot(1,3,1)
plt.imshow(img, cmap='gray_r')
plt.title(f"Original: {labels[10].item()}")

plt.subplot(1,3,2)
plt.imshow(cam, cmap='jet')
plt.title("CAM Heatmap")

plt.subplot(1,3,3)
plt.imshow(img, cmap='gray_r')
plt.imshow(cam, cmap='jet', alpha=0.6)
plt.title(f"Predicted: {pred_class}")
plt.show()


ModuleNotFoundError: No module named 'torch'

Colab link: https://colab.research.google.com/drive/1D3aS7LuLPtkBObK1S4m6UG2RM5RSnwYy?usp=sharing